In [54]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("../")

WAREHOUSE_DIR = (
    PROJECT_ROOT /
    "data" /
    "warehouse"
)

In [55]:
fact_orders = pd.read_csv(
    WAREHOUSE_DIR /
    "fact_orders.csv",
    parse_dates=[
        "order_purchase_timestamp"
    ]
)


dim_customer = pd.read_csv(
    WAREHOUSE_DIR /
    "dim_customer.csv"
)


dim_product = pd.read_csv(
    WAREHOUSE_DIR /
    "dim_product.csv"
)


dim_seller = pd.read_csv(
    WAREHOUSE_DIR /
    "dim_seller.csv"
)

In [56]:
fact_orders.head()

,order_id,customer_unique_id,product_id,seller_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_revenue,delivery_days,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,38.71,8.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,141.46,13.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,179.12,9.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,72.20,13.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,28.62,2.0,5.0


### Define Analysis Reference Date
- This is the cutoff date (latest transaction date) to know the last purchase's date

In [57]:
reference_date = (
    fact_orders[
        "order_purchase_timestamp"
    ]
    .max()
)


reference_date

Timestamp('2018-10-17 17:30:18')

### Customer Base

In [58]:
customers_ml = (

fact_orders
[
"customer_unique_id"
]
.drop_duplicates()
.reset_index(drop=True)

)

customers_ml = pd.DataFrame(
{
"customer_unique_id":
customers_ml
}
)

### RFM Features 
- Recency = How recently did the customer buy?
- Frequency = Number of orders
- Monetary = Total spending

In [59]:
recency = (

fact_orders

.groupby(
"customer_unique_id"
)

["order_purchase_timestamp"]

.max()

.reset_index()

)

recency["recency_days"] = (

reference_date -

recency[
"order_purchase_timestamp"
]

).dt.days

recency = recency[
[
"customer_unique_id",
"recency_days"
]
]

In [60]:
frequency = (

fact_orders

.groupby(
"customer_unique_id"
)

["order_id"]

.nunique()

.reset_index()

.rename(
columns={
"order_id":
"frequency"
}
)

)

In [61]:
monetary = (

fact_orders

.groupby(
"customer_unique_id"
)

["order_revenue"]

.sum()

.reset_index()

.rename(
columns={
"order_revenue":
"monetary"
}
)

)

In [62]:
# Merge them
customers_ml = (

customers_ml

.merge(
recency,
on="customer_unique_id",
how="left"
)

.merge(
frequency,
on="customer_unique_id",
how="left"
)

.merge(
monetary,
on="customer_unique_id",
how="left"
)

)

In [63]:
customers_ml.head()

,customer_unique_id,recency_days,frequency,monetary
0,7c396fd4830fd04220f754e42b4e5bff,380,2,82.82
1,af07308b275d755c9edb36a90c618231,84,1,141.46
2,3a653a41f6f9fc3d2a113cf8398680e8,70,1,179.12
3,7c142cf63193a1473d2e66489a9ae977,332,1,72.20
4,72632f0f9dd73dfee390c9b22eb56dd6,245,1,28.62


### Average Order Value 

In [64]:
avg_order_value = (

fact_orders

.groupby(
"customer_unique_id"
)

["order_revenue"]

.agg(
[
"mean",
"count"
]
)

.reset_index()

)

In [65]:
avg_order_value.columns = [

"customer_unique_id",

"avg_order_value",

"total_items"

]

In [66]:
customers_ml = customers_ml.merge(

avg_order_value,

on="customer_unique_id",

how="left"

)

### Product Behaviour Features

In [67]:
product_diversity = (

fact_orders

.groupby(
"customer_unique_id"
)

["product_id"]

.nunique()

.reset_index()

.rename(
columns={
"product_id":
"unique_products"
}
)

)

product_diversity = (

fact_orders

.groupby(
"customer_unique_id"
)

["product_id"]

.nunique()

.reset_index()

.rename(
columns={
"product_id":
"unique_products"
}
)

)

In [68]:
product_behavior = fact_orders.merge(

dim_product[
[
"product_key",
"product_category_name_english"
]
],

left_on="product_id",

right_on="product_key",

how="left"

)

category_diversity = (

product_behavior

.groupby(
"customer_unique_id"
)

[
"product_category_name_english"
]

.nunique()

.reset_index()

.rename(
columns={
"product_category_name_english":
"unique_categories"
}
)

)


customers_ml = customers_ml.merge(

category_diversity,

on="customer_unique_id",

how="left"

)

### Seller Behaviour Feature
- How many sellers did the customer purchase from

In [69]:
seller_diversity = (

fact_orders

.groupby(
"customer_unique_id"
)

["seller_id"]

.nunique()

.reset_index()

.rename(
columns={
"seller_id":
"unique_sellers"
}
)

)

customers_ml = customers_ml.merge(

seller_diversity,

on="customer_unique_id",

how="left"

)

### Customer Experience Features

#### Average Review Score

In [70]:
review_features = (

fact_orders

.groupby(
"customer_unique_id"
)

["review_score"]

.mean()

.reset_index()

.rename(
columns={
"review_score":
"avg_review_score"
}
)

)

In [71]:
customers_ml = customers_ml.merge(

review_features,

on="customer_unique_id",

how="left"

)

#### Delivery Problems - Late Delivery Ratio

In [72]:
fact_orders["is_late"] = (

pd.to_datetime(
fact_orders[
"order_delivered_customer_date"
]
)

>

pd.to_datetime(
fact_orders[
"order_estimated_delivery_date"
]
)

)

In [73]:
late_ratio = (

fact_orders

.groupby(
"customer_unique_id"
)

["is_late"]

.mean()

.reset_index()

.rename(
columns={
"is_late":
"late_delivery_ratio"
}
)

)

In [74]:
customers_ml = customers_ml.merge(

late_ratio,

on="customer_unique_id",

how="left"

)

### Geography Features

#### Customer Location

In [75]:
customer_geo = dim_customer[
[
"customer_unique_id",
"state",
"latitude",
"longitude"
]
]

In [76]:
customers_ml = customers_ml.merge(

customer_geo,

on="customer_unique_id",

how="left"

)

### Churn Label
- Customer inactive > 180 days = churn

In [77]:
CHURN_THRESHOLD = 180

customers_ml["churn_label"] = np.where(

customers_ml["recency_days"]

>

CHURN_THRESHOLD,

1,

0

)

### Missing Values Handling

In [78]:
customers_ml.isnull().sum()

customer_unique_id       0
recency_days             0
frequency                0
monetary                 0
avg_order_value        685
total_items              0
unique_categories        0
unique_sellers           0
avg_review_score       725
late_delivery_ratio      0
state                  278
latitude               278
longitude              278
churn_label              0
dtype: int64

In [79]:
numeric_columns = (

customers_ml

.select_dtypes(
include=np.number
)

.columns

)


customers_ml[numeric_columns] = (

customers_ml[numeric_columns]

.fillna(0)

)

In [80]:
categorical_columns = (

customers_ml

.select_dtypes(
exclude=np.number
)

.columns

)


customers_ml[categorical_columns] = (

customers_ml[categorical_columns]

.fillna("unknown")

)

### ML Feature Dataset Review

In [81]:
customers_ml.shape

(99441, 14)

In [82]:
customers_ml.head()

,customer_unique_id,recency_days,frequency,monetary,avg_order_value,total_items,unique_categories,unique_sellers,avg_review_score,late_delivery_ratio,state,latitude,longitude,churn_label
0,7c396fd4830fd04220f754e42b4e5bff,380,2,82.82,41.41,2,2,2,4.5,0.0,SP,-23.577482,-46.587077,1
1,7c396fd4830fd04220f754e42b4e5bff,380,2,82.82,41.41,2,2,2,4.5,0.0,SP,-23.577482,-46.587077,1
2,af07308b275d755c9edb36a90c618231,84,1,141.46,141.46,1,1,1,4.0,0.0,BA,-12.186877,-44.540232,0
3,3a653a41f6f9fc3d2a113cf8398680e8,70,1,179.12,179.12,1,1,1,5.0,0.0,GO,-16.745150,-48.514783,0
4,7c142cf63193a1473d2e66489a9ae977,332,1,72.20,72.20,1,1,1,5.0,0.0,RN,-5.774002,-35.270976,1


In [ ]:
customers_ml["churn_label"].value_counts(normalize=True)

churn_label
1    0.707243
0    0.292757
Name: proportion, dtype: float64

In [85]:
FEATURE_DIR = (

PROJECT_ROOT /
"data" /
"features"

)


FEATURE_DIR.mkdir(
parents=True,
exist_ok=True
)

customers_ml.to_csv(

FEATURE_DIR /
"customer_ml_features.csv",

index=False

)